<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Project_02_Mini_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
#Training Data

text = """
i love machine learning
machine learning is amazing
deep learning uses neural networks
transformers changed ai
i love transformers
ai is the future
"""

In [3]:
words=text.split()
vocab=sorted(set(words))
# vocab=sorted(set(text.split()))

# Create Word -> Number
word_to_idx={
    word : i
    for i ,word in enumerate(vocab)
}
word_to_idx



{'ai': 0,
 'amazing': 1,
 'changed': 2,
 'deep': 3,
 'future': 4,
 'i': 5,
 'is': 6,
 'learning': 7,
 'love': 8,
 'machine': 9,
 'networks': 10,
 'neural': 11,
 'the': 12,
 'transformers': 13,
 'uses': 14}

In [4]:
#Reverse the Dictionary later : number -> Word During generation
idx_to_word={
    i : word
    for  word,i in word_to_idx.items()
}
idx_to_word

{0: 'ai',
 1: 'amazing',
 2: 'changed',
 3: 'deep',
 4: 'future',
 5: 'i',
 6: 'is',
 7: 'learning',
 8: 'love',
 9: 'machine',
 10: 'networks',
 11: 'neural',
 12: 'the',
 13: 'transformers',
 14: 'uses'}

In [5]:
enumerate(vocab)

In [6]:
 word_to_idx

{'ai': 0,
 'amazing': 1,
 'changed': 2,
 'deep': 3,
 'future': 4,
 'i': 5,
 'is': 6,
 'learning': 7,
 'love': 8,
 'machine': 9,
 'networks': 10,
 'neural': 11,
 'the': 12,
 'transformers': 13,
 'uses': 14}

In [7]:
idx_to_word

{0: 'ai',
 1: 'amazing',
 2: 'changed',
 3: 'deep',
 4: 'future',
 5: 'i',
 6: 'is',
 7: 'learning',
 8: 'love',
 9: 'machine',
 10: 'networks',
 11: 'neural',
 12: 'the',
 13: 'transformers',
 14: 'uses'}

In [8]:
sequence_length = 3

inputs = []
targets = []

tokenized = [
    word_to_idx[word]
    for word in words
]

In [9]:
len(tokenized)

23

In [10]:


for i in range(len(tokenized)-sequence_length):

    input_seq = tokenized[i:i+sequence_length]

    target = tokenized[i+sequence_length]

    inputs.append(input_seq)

    targets.append(target)

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

In [11]:
# class MiniGPT(nn.Module):

#     def __init__(
#         self,
#         vocab_size,
#         embed_dim,
#         num_heads,
#         hidden_dim
#     ):

#         super().__init__()

#         # embeddings
#         self.embedding = nn.Embedding(
#             vocab_size,
#             embed_dim
#         )

#         # attention
#         self.attention = nn.MultiheadAttention(
#             embed_dim,
#             num_heads,
#             batch_first=True
#         )

#         # feed forward
#         self.ff = nn.Sequential(
#             nn.Linear(embed_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, embed_dim)
#         )

#         # final prediction
#         self.fc = nn.Linear(
#             embed_dim,
#             vocab_size
#         )

#     def forward(self, x):

#         # embeddings
#         x = self.embedding(x)

#         # self attention
#         attn_output, _ = self.attention(
#             x,
#             x,
#             x
#         )

#         # feed forward
#         x = self.ff(attn_output)

#         # take last token
#         x = x[:, -1, :]

#         # predict next word
#         out = self.fc(x)

#         return out

class MiniGPT(nn.Module):
  def __init__(self, vocab_size,embed_dim,num_heads,hidden_dim):
     super().__init__()

     #embedding
     self.embedding=nn.Embedding(
         vocab_size,
         embed_dim
     )

     self.attention=nn.MultiheadAttention(
         embed_dim,
         num_heads,
         batch_first=True
     )

     self.ff=nn.Sequential(
         nn.Linear(embed_dim,hidden_dim),
         nn.ReLU(),
         nn.Linear(hidden_dim,embed_dim)
     )
     self.fc=nn.Linear(
         embed_dim,
         vocab_size
     )

  def forward(self,x):
    x=self.embedding(x)

    attn_output, _=self.attention(x,x,x)

    x=self.ff(attn_output)


    x = x[:, -1, :]

    out=self.fc(x)

    return out



In [12]:
model = MiniGPT(
    vocab_size=len(vocab),
    embed_dim=32,
    num_heads=2,
    hidden_dim=64
)

In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

In [14]:
for epoch in range(300):

    optimizer.zero_grad()

    outputs = model(inputs)

    loss = criterion(
        outputs,
        targets
    )

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item()}"
        )

Epoch 0, Loss: 2.7129271030426025
Epoch 50, Loss: 2.5033648398675723e-06
Epoch 100, Loss: 1.7881384906104358e-07
Epoch 150, Loss: 1.4305109630186053e-07
Epoch 200, Loss: 1.251697341331237e-07
Epoch 250, Loss: 1.0728833643725011e-07


In [15]:
def generate_text(start_text, length=5):

    model.eval()

    words_input = start_text.split()

    for _ in range(length):

        input_seq = [
            word_to_idx[word]
            for word in words_input[-3:]
        ]

        input_tensor = torch.tensor(
            [input_seq]
        )

        with torch.no_grad():

            output = model(input_tensor)

            predicted_idx = torch.argmax(
                output,
                dim=1
            ).item()

        predicted_word = idx_to_word[
            predicted_idx
        ]

        words_input.append(predicted_word)

    return " ".join(words_input)

In [16]:
print(
    generate_text("i love")
)

i love transformers ai is the future


In [17]:
print(
    generate_text("deep learning")
)

deep learning neural networks transformers changed ai


In [18]:
print(
    generate_text("amazing")
)

amazing deep uses neural networks transformers


## Now upgrading to GPT


*   Adding POSITIONAL ENCODING + CAUSAL MASKING




In [19]:
class BetterGPT(nn.Module):
  def __init__(self,vocab_size,embed_dim,num_heads,hidden_dim,max_len=100):
      super().__init__()

      self.embedding=nn.Embedding(
          vocab_size,
          embed_dim
      )

      self.position_embedding=nn.Embedding(
          max_len,
          embed_dim
      )

      self.attn=nn.MultiheadAttention(
        embed_dim,
        num_heads,
        batch_first=True
      )

      self.ff=nn.Sequential(
          nn.Linear(embed_dim,hidden_dim),
          nn.ReLU(),
          nn.Linear(hidden_dim,embed_dim)
      )

      self.fc=nn.Linear(
          embed_dim,
          vocab_size
      )
  def feed_forward(self,x):
      seq_len=x.size(1)

      positions=torch.arange(0,seq_len).unsqueeze(0)

      x=self.embedding(x) + self.position_embedding(positions)

      mask=torch.triu(torch.ones(seq_len,seq_len),diagonal=1).bool()

      attn_output, _ = self.attention(
            x,
            x,
            x,
            attn_mask=mask
        )

      x = self.ff(attn_output)

      x = x[:, -1, :]

      out = self.fc(x)

      return out







In [20]:
better_model = MiniGPT(
    vocab_size=len(vocab),
    embed_dim=32,
    num_heads=2,
    hidden_dim=64
)

In [21]:
criterion_i = nn.CrossEntropyLoss()

optimizer_i = optim.Adam(
    model.parameters(),
    lr=0.01
)

In [22]:
for epoch in range(300):

    optimizer_i.zero_grad()

    outputs = model(inputs)

    loss = criterion_i(
        outputs,
        targets
    )

    loss.backward()

    optimizer_i.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item()}"
        )

Epoch 0, Loss: 8.940695295223122e-08
Epoch 50, Loss: 1.5437366300830035e-06
Epoch 100, Loss: 6.556467724294635e-07
Epoch 150, Loss: 3.099432035469363e-07
Epoch 200, Loss: 2.0265538580588327e-07
Epoch 250, Loss: 1.4901138456480112e-07


In [23]:
def generate_text_i(start_text, length=5):

    better_model.eval()

    words_input_i = start_text.split()

    for _ in range(length):

        input_seq = [
            word_to_idx[word]
            for word in words_input_i[-3:]
        ]

        input_tensor = torch.tensor(
            [input_seq]
        )

        with torch.no_grad():

            output = better_model(input_tensor)

            predicted_idx = torch.argmax(
                output,
                dim=1
            ).item()

        predicted_word = idx_to_word[
            predicted_idx
        ]

        words_input_i.append(predicted_word)

    return " ".join(words_input_i)

In [24]:
print(
    generate_text_i("networks")
)

networks future is the is the


In [26]:
print(
    generate_text_i("machine learning is")
)

machine learning is the the ai is is
